# Scrollie — interactive segmentation viewer for Dafne and Dafne plus MedSAM

Pick a file and scroll through slices comparing:
- **Left**: original fat-fraction image
- **Centre**: Dafne segmentation overlay
- **Right**: Dafne + MedSAM segmentation overlay (if available)

In [1]:
import glob
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import SimpleITK as sitk
from ipywidgets import interact, IntSlider, Dropdown, VBox, HBox
import ipywidgets as widgets
from IPython.display import display

In [2]:
DAFNE_DIR   = "dafne_thigh_results"
MEDSAM_DIR  = "dafne_medsam_results"
IMAGE_BASE  = "myosegmenTUM"

# find all dafne npz files and build a label -> paths dict
npz_files = sorted(glob.glob(os.path.join(DAFNE_DIR, "*.npz")))

def npz_to_nii(npz_path):
    """Reconstruct the original NIfTI path from a dafne npz filename."""
    fname   = os.path.basename(npz_path)                         # HV001_1_FATFRACTION_stack1_dafne_thigh.npz
    subject = fname.split("_FATFRACTION")[0]                     # HV001_1
    stack   = re.search(r"(FATFRACTION_stack\d+)", fname).group(1)  # FATFRACTION_stack1
    return os.path.join(IMAGE_BASE, subject, "ImageData",
                        f"{subject}_{stack[:-len('_stack1')]}",   # folder: HV001_1_FATFRACTION  
                        f"{subject}_{stack}.nii")                 # file:   HV001_1_FATFRACTION_stack1.nii

def npz_to_nii(npz_path):
    fname   = os.path.basename(npz_path)
    subject = fname.split("_FATFRACTION")[0]
    m       = re.search(r"stack(\d+)", fname)
    stack_n = m.group(1)
    return os.path.join(IMAGE_BASE, subject, "ImageData",
                        f"{subject}_FATFRACTION",
                        f"{subject}_FATFRACTION_stack{stack_n}.nii")

file_options = {os.path.basename(p).replace("_dafne_thigh.npz", ""): p for p in npz_files}
print(f"Found {len(file_options)} segmented stacks")

Found 53 segmented stacks


In [3]:
def build_overlay(segmentation, label_map, cmap):
    """Build an RGBA overlay array from a labelled segmentation volume."""
    overlay = np.zeros((*segmentation.shape, 4), dtype=float)
    for i, (label_idx, name) in enumerate(label_map.items()):
        color = cmap(i)
        overlay[segmentation == label_idx] = [color[0], color[1], color[2], 0.5]
    return overlay

def load_stack(npz_path):
    """Return (gt_norm, segmentation, label_map, overlay) for one npz file."""
    nii_path = npz_to_nii(npz_path)
    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(float)
    gt_norm   = (img_array - img_array.min()) / (img_array.max() - img_array.min() + 1e-8)

    data      = np.load(npz_path)
    masks     = dict(data)
    label_map = {i: name for i, name in enumerate(masks.keys(), start=1)}
    n_slices, H, W = next(iter(masks.values())).shape
    segmentation = np.zeros((n_slices, H, W), dtype=np.uint16)
    for label_idx, (name, arr) in enumerate(masks.items(), start=1):
        segmentation[arr > 0] = label_idx

    cmap    = plt.colormaps["tab20"].resampled(len(label_map))
    overlay = build_overlay(segmentation, label_map, cmap)
    return gt_norm, overlay, label_map, cmap

In [6]:
# --- widgets ---
file_dropdown = Dropdown(options=list(file_options.keys()), description="Stack:")
slice_slider  = IntSlider(min=0, max=1, step=1, value=0, description="Slice:",
                           layout=widgets.Layout(width="600px"))
out = widgets.Output()

# cache so we don't reload on every slice move
_cache = {}

def get_data(label):
    if label not in _cache:
        npz_path = file_options[label]
        gt_norm, overlay_dafne, label_map, cmap = load_stack(npz_path)

        # try medsam version
        medsam_npz = os.path.join(MEDSAM_DIR,
                                   os.path.basename(npz_path).replace("_dafne_thigh", "_dafne_medsam"))
        if os.path.exists(medsam_npz):
            _, overlay_medsam, _, _ = load_stack(medsam_npz)
        else:
            overlay_medsam = None

        legend_patches = [
            mpatches.Patch(color=cmap(i), alpha=0.6, label=name)
            for i, (_, name) in enumerate(label_map.items())
        ]
        _cache[label] = (gt_norm, overlay_dafne, overlay_medsam, legend_patches)
        slice_slider.max = gt_norm.shape[0] - 1
        slice_slider.value = 0
    return _cache[label]

def on_file_change(change):
    _cache.clear()
    get_data(change["new"])
    render(file_dropdown.value, slice_slider.value)

def render(label, slice_idx):
    gt_norm, overlay_dafne, overlay_medsam, legend_patches = get_data(label)
    n_panels = 3 if overlay_medsam is not None else 2
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))

    img = gt_norm[slice_idx]

    axes[0].imshow(img, cmap="gray", origin="lower")
    axes[0].set_title(f"Image — slice {slice_idx}")
    axes[0].axis("off")

    axes[1].imshow(img, cmap="gray", origin="lower")
    axes[1].imshow(overlay_dafne[slice_idx], origin="lower")
    axes[1].set_title("Dafne")
    axes[1].axis("off")
    axes[1].legend(handles=legend_patches, loc="lower right", fontsize=6, framealpha=0.7)

    if overlay_medsam is not None:
        axes[2].imshow(img, cmap="gray", origin="lower")
        axes[2].imshow(overlay_medsam[slice_idx], origin="lower")
        axes[2].set_title("Dafne + MedSAM")
        axes[2].axis("off")

    fig.suptitle(label, fontsize=10)
    plt.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()

def on_slice_change(change):
    render(file_dropdown.value, change["new"])

file_dropdown.observe(on_file_change, names="value")
slice_slider.observe(on_slice_change, names="value")

# initial render
if file_options:
    get_data(file_dropdown.value)
    render(file_dropdown.value, 0)

display(VBox([file_dropdown, slice_slider, out]))